In [34]:
class PasswordValidator():
    OPTIONS = {
        'min_len': 8,
        'contain_numbers': False,
        }
    # BEGIN (write your solution here)

    # def __init__(self, **args, 
                #  min_len=OPTIONS['min_len'], contain_numbers=OPTIONS['contain_numbers']
        # ):
        # self.__min_len__ = min_len
        # self.__contain_numbers__ = contain_numbers
        # print(list(args))
        # if item in list(args) not in OPTIONS.keys():
        #     pass
        # if OPTIONS.k in list(args):
        #     pass
    def __init__(self, **options):
        self.options = PasswordValidator.OPTIONS | options
        print(options)        

    def validate(self, password):
        errors = {}
        
        if len(password) < self.options['min_len']:
            errors['min_len'] = 'too small'

        if self.options['contain_numbers']:
            if not any(char.isdigit() for char in password):
                errors['contain_numbers'] = 'should contain at least one number'

        return errors
    # END



# validator = PasswordValidator()
# errors = validator.validate('qwerty1')
# print(errors)  # => {'min_len': 'too small'}


options = {'contain_numbers': True}
validator = PasswordValidator(**options)
errors = validator.validate('qwerty')
# print(errors)


# валидатор должен игнорировать несуществующие опции
# validator = PasswordValidator(numberz=None)
# errors = validator.validate('qwertya3sdf')
# print(errors) # => {}

{'contain_numbers': True}


In [38]:
{
    'min_len': 8,
    'contain_numbers': False,
} | {'numberz'}

TypeError: unsupported operand type(s) for |: 'dict' and 'set'

## Изменяемая конфигурация 

In [97]:
class Truncater:
    OPTIONS = {
        'separator': '...',
        'length': 200,
    }
    def __init__(self, **options):
        self.options = {**self.OPTIONS, **options}
        self.__options = {**self.OPTIONS, **options}


    def truncate(self, term, **options):
        self.options = {**self.options, **options}
        if len(term) <= self.options['length']:
            # result = term[:self.options['length']]
            result = term
        else:
            result = f"{term[:self.options['length']]}{self.options['separator']}"
        
        self.options = self.__options
        return result


truncater = Truncater(length=3)
assert truncater.truncate('one two') == 'one...'
print(truncater.truncate('one two', separator='!'))
assert truncater.truncate('one two', separator='!') == 'one!'
assert truncater.truncate('one two') == 'one...'
assert truncater.truncate('one two', length=7) == 'one two'

truncater = Truncater()
assert truncater.truncate('one two') == 'one two'
assert truncater.truncate('one two', length=6) == 'one tw...'
assert truncater.truncate('one two', separator='.') == 'one two'
assert truncater.truncate('one two', separator='.')

one!


Example solution

In [98]:
# BEGIN
class Truncater():
    OPTIONS = {
        'separator': '...',
        'length': 200,
    }

    def __init__(self, **options):
        self.options = {**self.OPTIONS, **options}

    def truncate(self, text, **options):
        current_options = {**self.options, **options}
        if len(text) <= current_options['length']:
            return text
        substr = text[:current_options['length']]
        return f"{substr}{current_options['separator']}"
# END

In [ ]:
# BEGIN
class Url():
    def __init__(self, url):
        self.url = urlparse(url)
        self.query_params = {}

        if self.url.query:
            self.query_params = parse_qs(self.url.query)

    def get_scheme(self):
        return self.url.scheme

    def get_hostname(self):
        return self.url.hostname

    def get_query_params(self):
        return self.query_params

    def get_query_param(self, key, default_value=None):
        return self.query_params.get(key, [default_value])[0]

    def __eq__(self, other):
        return self.url == other.url
# END

## Fluent interface

In [99]:
raw = [{'name': 'istambul', 'country': 'turkey'},
       {'name': 'Moscow ', 'country': ' Russia'},
       {'name': 'iStambul', 'country': 'tUrkey'},
       {'name': 'antalia', 'country': 'turkeY '},
       {'name': 'samarA', 'country': '  ruSsiA'}]


In [102]:
from functools import reduce as _reduce


class Collection:
    def __init__(self, iterable):
        self.iterable = iterable

    def map_(self, func):
        return Collection(list(map(func, self.iterable)))

    def filter_(self, func):
        return Collection(list(filter(func, self.iterable)))

    def reduce_(self, func, acc=None):
        return Collection([_reduce(func, self.iterable, acc)])

    # возвращает коллекцию с уникальными значениями
    def unique(self):
        tuples = set(tuple(sorted(d.items())) for d in self.iterable)
        return Collection(list(dict(t) for t in tuples))

    # группирует коллекцию по указаному ключу
    def group_by(self, func):
        def reducer(acc, val):
            key, value = func(val)
            if key not in acc:
                acc[key] = []
            acc[key].append(value)
            return acc
        result_dict = _reduce(reducer, self.iterable, {})
        return Collection([{k: v} for k, v in result_dict.items()])

    # сортирует колекцию по ключу !!!!!! здесь не ключ а функция 
    def sort_by(self, func):
        return Collection(sorted(self.iterable, key=func))

    def print(self):
        print(self.iterable)
        return Collection(self.iterable)

    def all(self):
        return list(self.iterable)

In [ ]:

c.unique().group_by(lambda row: (row['age'], row['name'])).all()
# [{30: ['Charlie']}, {20: ['Bob', 'Alice']}]

c.unique().group_by(lambda row: (row['age'], row['name'])).sort_by(lambda row: list(row.keys())).all()
# [{20: ['Bob', 'Alice']}, {30: ['Charlie']}]

In [209]:
def format(raw):
    raw_c = Collection([{ k: v.strip().lower() for  k, v in item.items() } for item in raw])

    return raw_c.unique(). \
        sort_by(lambda d: d['name']). \
        sort_by(lambda d: d['country']). \
        group_by(lambda row: (row['country'], row['name'])). \
        all()

res = format(raw)
print(res)


''' another solution

# BEGIN
def format(data):
    c = Collection(data)
    return c.map_(_normalise) \
        .unique() \
        .group_by(lambda row: (row['country'], row['name'])) \
        .map_(lambda row: {key: sorted(values) for key, values in row.items()}) \
        .sort_by(lambda row: list(row.keys())) \
        .all()


def _normalise(row):
    return {'name': row['name'].lower().strip(), 'country': row['country'].lower().strip()}
# END

'''

[{'russia': ['moscow', 'samara']}, {'turkey': ['antalia', 'istambul']}]


In [191]:
raw = [{'name': 'istambul', 'country': 'turkey'},
       {'name': 'Moscow ', 'country': ' Russia'},
       {'name': 'iStambul', 'country': 'tUrkey'},
       {'name': 'antalia', 'country': 'turkeY '},
       {'name': 'samarA', 'country': '  ruSsiA'}]
res = format(raw)
expected = [{'russia': ['moscow', 'samara']},
            {'turkey': ['antalia', 'istambul']}]

assert res == expected

## Builders

In [259]:
from datetime import datetime, timedelta


class Booking:
    RESERVED = set()

    def __init__(self) -> None:
        self.reserved = Booking.RESERVED

    def book(self, start_date, finish_date):
        start = datetime.strptime(start_date, "%Y-%m-%d")
        end = datetime.strptime(finish_date, "%Y-%m-%d")
        if end <= start:
            return False
        
        date_generated = {start + timedelta(days=x) for x in range(0, (end-start).days)}

        if self.reserved & date_generated:
            return False
        
        self.reserved |= date_generated
        return True


def test_booking():
    booking = Booking()

    assert not booking.book('2008-11-10', '2008-11-05')
    assert booking.book('2008-11-11', '2008-11-13')
    # booking.book('2008-11-11', '2008-11-13')
    assert not booking.book('2008-11-12', '2008-11-12')
    assert not booking.book('2008-11-12', '2008-11-14')
    assert booking.book('2008-11-10', '2008-11-11')
    assert not booking.book('2008-11-12', '2008-11-13')
    assert not booking.book('2008-11-13', '2008-11-13')
    assert booking.book('2008-11-13', '2008-11-14')
    assert booking.book('2008-05-08', '2008-05-18')
    assert not booking.book('2008-05-09', '2008-05-10')

test_booking()


example

In [ ]:
# BEGIN
from datetime import date

class Booking():
    def __init__(self):
        self.dates = []

    def book(self, begin, end):
        new_begin = date.fromisoformat(begin)
        new_end = date.fromisoformat(end)
        if self._can_book(new_begin, new_end):
            self.dates.append((new_begin, new_end))
            return True
        return False

    def _can_book(self, begin, end):
        if begin >= end:
            return False
        for booked_begin, booked_end in self.dates:
            if begin < booked_end and end > booked_begin:
                return False
        return True
# END

In [246]:
s1 = {1, 2}
s2 = {3}

if s1 & s2:
    print('Forbidden')
else:
    print('Ok')

s1 |= s2


Ok


In [247]:
s1

{1, 2, 3}

In [232]:
import datetime 
start = datetime.datetime.strptime("21-06-2014", "%d-%m-%Y")
end = datetime.datetime.strptime("07-07-2014", "%d-%m-%Y")
date_generated = {start + datetime.timedelta(days=x) for x in range(0, (end-start).days)}

for date in date_generated:
    print(date.strftime("%d-%m-%Y"))

21-06-2014
26-06-2014
22-06-2014
28-06-2014
01-07-2014
05-07-2014
06-07-2014
02-07-2014
24-06-2014
29-06-2014
30-06-2014
27-06-2014
03-07-2014
04-07-2014
23-06-2014
25-06-2014


In [250]:

def test_booking():
    booking = Booking()

    assert not booking.book('2008-11-10', '2008-11-05')
    assert booking.book('2008-11-11', '2008-11-13')
    assert not booking.book('2008-11-12', '2008-11-12')
    assert not booking.book('2008-11-12', '2008-11-14')
    assert booking.book('2008-11-10', '2008-11-11')
    assert not booking.book('2008-11-12', '2008-11-13')
    assert not booking.book('2008-11-13', '2008-11-13')
    assert booking.book('2008-11-13', '2008-11-14')
    assert booking.book('2008-05-08', '2008-05-18')
    assert not booking.book('2008-05-09', '2008-05-10')

test_booking()

AssertionError: 

In [217]:
from datetime import datetime

#  == datetime.strptime('2008-11-10', "%Y-%m-%d")

True

In [220]:
s = set()
s.add(datetime.strptime('2008-11-10', "%Y-%m-%d"))

In [222]:
s.add(datetime.strptime('2008-11-11', "%Y-%m-%d"))

In [223]:
s

{datetime.datetime(2008, 11, 10, 0, 0), datetime.datetime(2008, 11, 11, 0, 0)}

In [224]:
datetime.strptime('2008-11-10', "%Y-%m-%d") in s 

True

In [233]:
# date_generated = [start + datetime.timedelta(days=x) for x in range(0, (end-start).days)]

import datetime 
start = datetime.datetime.strptime("21-06-2014", "%d-%m-%Y")
end = datetime.datetime.strptime("07-07-2014", "%d-%m-%Y")
date_generated = {start + datetime.timedelta(days=x) for x in range(0, (end-start).days)}

for date in date_generated:
    print(date.strftime("%d-%m-%Y"))

21-06-2014
26-06-2014
22-06-2014
28-06-2014
01-07-2014
05-07-2014
06-07-2014
02-07-2014
24-06-2014
29-06-2014
30-06-2014
27-06-2014
03-07-2014
04-07-2014
23-06-2014
25-06-2014


In [234]:
type(date_generated)

set